In [1]:
import torch
import time
from transformers import MarianMTModel, MarianTokenizer

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Opus-MT Arabic to German
model_name = "Helsinki-NLP/opus-mt-ar-de"
opus_tokenizer = MarianTokenizer.from_pretrained(model_name)
opus_model = MarianMTModel.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

C:\Users\aksha\anaconda3\envs\lingomatch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-ar-de. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


source.spm:   0%|          | 0.00/903k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/829k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

C:\Users\aksha\anaconda3\envs\lingomatch\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

In [4]:
print(f"Loaded {model_name}")
allocated_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used: {allocated_gb:.2f} GB")

Loaded Helsinki-NLP/opus-mt-ar-de
GPU memory used: 0.31 GB


In [5]:
test_sentences = [
    "مرحبا، كيف حالك؟",
    "اسمي محمد علي، ولدت في حلب عام 1987",
    "أسكن في شارع الرشيد رقم 42، دمشق",
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    "رقم الهوية الشخصية: 0123456789",
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    "بناءً على المادة الثانية من قانون العقوبات",
    "السيد عبد الرحمن الحسيني",
    "إن شاء الله سأصل غدا في المساء",
]

print(f"Opus-MT-ar-de on {len(test_sentences)} test sentences\n")
start = time.time()

for i, ar_text in enumerate(test_sentences, 1):
    inputs = opus_tokenizer(ar_text, return_tensors="pt", padding=True).to(device)
    translated = opus_model.generate(**inputs, max_length=128)
    de_text = opus_tokenizer.decode(translated[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

elapsed = time.time() - start
print(f"Total: {elapsed:.2f}s ({elapsed/len(test_sentences):.2f}s per sentence)")

Opus-MT-ar-de on 10 test sentences

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammad Ali. Ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der guten Straße 42. Straße, Damaskus.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist eine offizielle Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Versionsdatum: 15 Ramadan 1444 verlassen

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Identitätsnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Nach Artikel 2 des Strafgesetzbuches.

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdelbar al-Hassini.

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Wenn Gott will, werde ich morgen Abend da sein.

Total: 4.10s (0.41s per sentence)


In [6]:
import sacrebleu

references_for_bleu = [
    {"ar": "اسمي أحمد وأنا من سوريا", "ref_de": "Ich heiße Ahmed und ich komme aus Syrien."},
    {"ar": "أسكن في شارع الرشيد رقم 42، دمشق", "ref_de": "Ich wohne in der Rashid-Straße Nummer 42 in Damaskus."},
    {"ar": "رقم الهوية الشخصية: 0123456789", "ref_de": "Personalausweisnummer: 0123456789"},
]

hypotheses = []
references = []

for item in references_for_bleu:
    inputs = opus_tokenizer(item["ar"], return_tensors="pt", padding=True).to(device)
    out = opus_model.generate(**inputs, max_length=128)
    de = opus_tokenizer.decode(out[0], skip_special_tokens=True)
    hypotheses.append(de)
    references.append(item["ref_de"])
    print(f"AR:  {item['ar']}")
    print(f"REF: {item['ref_de']}")
    print(f"HYP: {de}")
    print()

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f"BLEU (Opus-MT-ar-de): {bleu.score:.2f}")
print(f"BLEU (NLLB-1.3B, from notebook 06): 42.36")
print(f"BLEU (NLLB-600M, from week 1): 38.32")

AR:  اسمي أحمد وأنا من سوريا
REF: Ich heiße Ahmed und ich komme aus Syrien.
HYP: Mein Name ist Ahmed und ich komme aus Syrien.

AR:  أسكن في شارع الرشيد رقم 42، دمشق
REF: Ich wohne in der Rashid-Straße Nummer 42 in Damaskus.
HYP: Ich wohne in der guten Straße 42. Straße, Damaskus.

AR:  رقم الهوية الشخصية: 0123456789
REF: Personalausweisnummer: 0123456789
HYP: Identitätsnummer: 0123456789

BLEU (Opus-MT-ar-de): 43.81
BLEU (NLLB-1.3B, from notebook 06): 42.36
BLEU (NLLB-600M, from week 1): 38.32
